# Session 2a — Building agents with visual builders

**Building an AI Research Team in n8n** 

Across this tutorial we build an *in silico research team* around one clinical question:

> **Which patients develop heart failure after an acute myocardial infarction (MI)?**

In this module you build the first teammates in **n8n**, a visual workflow builder, without writing agent code:

| Step | What it does | What it shows |
|---|---|---|
| **Literature Scout** (agent) | Searches PubMed and summarizes what is known about heart failure after MI | Tools as API calls |
| **Human Review** (you) | Approves or declines the Scout's findings | A human checkpoint |
| **Data Analyst** (agent) | Checks the approved findings against our patient data with a tool, then proposes features and predictive questions | Passing one agent's work to the next, and a tool that is code, not an API call |

The workflow ends with a report you can save as a PDF.

**Ready-made workflows are included.** The `workflows/` folder has finished versions that you can import into n8n to have the whole team running in minutes, instead of (or alongside) building it by hand. See *Shortcut: import a ready-made workflow* in Section 2.

> **What is an agent?** A language model that is given *instructions* and *tools*, decides for itself which tool to call, reads the result, and repeats until the task is done.

**By the end you should be able to:**
- Find the components of an agent (model, instructions, tools, memory) on a canvas.
- Explain that a *tool* is a described capability (an API call or a piece of code), and that the model decides when to use it.
- Chain two agents and add a human approval between them.
- Decide when a task needs an agent at all.

*Please install n8n before the session.

---

## Before the session

| Item | Notes |
|---|---|
| Docker Desktop **or** an n8n Cloud account | Docker runs n8n on your computer; Cloud needs nothing installed (see the next section) |
| LLM access | An OpenAI, Anthropic, or Azure OpenAI key with models deployed that supports tool calling |
| Python 3.10+ | With packages installed in requirements.txt |
| Tutorial data | `Data/mi_heart_failure.csv` and `Data/dict_heart_failure.json` |
| Ready-made workflows | `workflows/mi_research_team.json` and `workflows/mi_research_team_mcp.json`: import them into n8n to have the team running (see Section 2) |

The data is public and de-identified. Never paste identifiable patient data into a hosted LLM service. Set up n8n as described next, then run the check cell below it.

### Get n8n running before the session

You need n8n open in your browser at the start. Choose **one** of these:

**Option A — Docker (free, runs on your computer; recommended)**
1. Install [Docker Desktop](https://docs.docker.com/get-docker/) (Mac, Windows, or Linux) and start it.
2. In a terminal, run this command **once**. It downloads n8n and creates the container (Docker creates the `n8n_data` volume for your data automatically):
   ```bash
   docker run -d --name n8n -p 5678:5678 -v n8n_data:/home/node/.n8n docker.n8n.io/n8nio/n8n
   ```
3. Give it a minute, then open http://localhost:5678 and create the owner account (any email and password; they stay on your computer). If the page does not load yet, open the **Logs** tab of the n8n container in Docker Desktop to see whether it is still starting.
4. **From now on, start and stop n8n in Docker Desktop.** Open the **Containers** tab, find **n8n**, and use the ▶ (start) and ■ (stop) buttons. Open n8n with the `5678:5678` link in that row, or go to http://localhost:5678. Your workflows and credentials are kept in the `n8n_data` volume, so they are still there next time. Do not run the command in step 2 again: it would fail with "name already in use". (From a terminal, the same thing is `docker start n8n` and `docker stop n8n`.)

**Option B — n8n Cloud (nothing to install)**
1. Go to [n8n.io/pricing](https://n8n.io/pricing), click **Start free trial**, and sign up with your email. No credit card is required.
2. Pick a workspace name. n8n gives you your own web address, and you build the workflow there.
3. The trial is time-limited (check the current length on the pricing page) and has limits, published as 1,000 executions and 180 seconds per execution. That should be enough for this tutorial, but a very long run could hit the 180-second limit.

The check below reports `n8n running on :5678` for Docker only. If you use Cloud, ignore that line.

In [23]:
# check environment and dependencies
import importlib.util, shutil, socket
from pathlib import Path


def port_open(port, host="localhost"):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

checks = {
    "pandas installed":                     importlib.util.find_spec("pandas") is not None,
    "scipy installed":                      importlib.util.find_spec("scipy") is not None,
    "requests installed":                   importlib.util.find_spec("requests") is not None,
    "Data/mi_heart_failure.csv found":                Path("Data/mi_heart_failure.csv").exists(),
    "Data/dict_heart_failure.json found":             Path("Data/dict_heart_failure.json").exists(),
    "docker on PATH":                       shutil.which("docker") is not None,
    "node on PATH (only for npm installs)": shutil.which("node") is not None,
    "n8n running on :5678":                 port_open(5678),
}
for name, ok in checks.items():
    print(f"{'✅' if ok else '⬜'}  {name}")

def show(name):
    # Print a prompt file from materials/prompts/ so you can copy it into n8n
    path = Path("materials/prompts") / name
    if not path.exists():
        print(f"{path} not found. Run every code cell in Section 1, from top to bottom, then try again.")
        return
    print(path.read_text())

✅  pandas installed
✅  scipy installed
✅  requests installed
✅  Data/mi_heart_failure.csv found
✅  Data/dict_heart_failure.json found
✅  docker on PATH
✅  node on PATH (only for npm installs)
✅  n8n running on :5678


## 1. The data: heart failure after MI

> **Before we build the workflow, we explore the data statistically.** This section runs in Python, by you, not by an agent. Its results become the *data profile*: the Data Analyst agent will look them up through a tool later, and you can check its numbers against the profile.

We use the **Myocardial infarction complications** dataset from the UCI Machine Learning Repository (ID 579): 1,700 patients admitted with acute MI to one hospital in Krasnoyarsk, Russia (1992–1995). We keep one outcome, **`heart_failure`** (chronic heart failure, 1 = yes, 0 = no).

- `Data/mi_heart_failure.csv`: one row per patient, with `patient_id`, 111 features recorded at admission and over the first three days, and the outcome.
- `Data/dict_heart_failure.json`: the data dictionary (description, codes, units, missing data, and the original UCI name of each column).

**Why explore first?** A language model cannot reliably compute over a 1,700 × 113 table. So we do the calculations here in Python, where the numbers are correct, and give the Data Analyst a tool that *looks up* the results. The agent decides what to look up, and reasons over the answers and the literature. (In Module 2, agents run their own analysis code.)

The cells below explore the data, then create `materials/` with the data profile, a review checklist for judging the agent, and every prompt you will paste into n8n.

### What is in the data

| Group | Columns | Examples |
|---|---|---|
| Demographics | 2 | `age`, `sex` |
| History before this MI | 31 | `prior_heart_failure`, `diabetes`, `hypertension_stage` |
| Condition on arrival | 10 | `systolic_bp_icu`, `cardiogenic_shock_at_admission` |
| Infarct location | 5 | `anterior_mi`, `inferior_mi` |
| ECG at admission | 26 | `ecg_rhythm_atrial_fibrillation` (all start with `ecg_`) |
| Clot-dissolving drugs | 7 | `fibrinolysis_streptase` (all start with `fibrinolysis_`) |
| Blood tests | 9 | `serum_potassium`, `esr` |
| Time to hospital | 1 | `time_to_hospital` |
| First three days | 20 | `pain_relapse_day_1`, `aspirin_icu` |
| **Outcome** | 1 | `heart_failure` |

Column names are plain-English versions of the original (Russian) UCI abbreviations; the dictionary lists each `original_name` and the codes. A few columns are mostly empty: `serum_cpk` (99.8%), `family_history_coronary_disease` (95.8%), and the emergency team's blood pressure (63.3%).

In [24]:
from pathlib import Path
import json
import pandas as pd

DATA = Path("Data")
MATERIALS = Path("materials")
PROMPTS = MATERIALS / "prompts"
PROMPTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA / "mi_heart_failure.csv")
dictionary = {col["name"]: col for col in json.loads((DATA / "dict_heart_failure.json").read_text())["columns"]}

# Quick checks: one row per patient, every column documented, a yes/no outcome
assert df["patient_id"].is_unique, "Duplicate patient IDs"
assert list(df.columns) == list(dictionary), "CSV columns and data dictionary do not match"
assert set(df["heart_failure"].unique()) == {0, 1}, "Outcome should be coded 0/1"

features = [name for name, col in dictionary.items() if col["role"] == "feature"]
n_hf = int(df["heart_failure"].sum())
print(f"{len(df):,} patients, {len(features)} features, outcome = heart_failure")
print(f"Heart failure: {n_hf} patients ({n_hf / len(df):.1%})")
print("Sample of the first 5 patients:")
df[["patient_id", "age", "sex", "prior_heart_failure", "diabetes", "anterior_mi", "heart_failure"]].head()

1,700 patients, 111 features, outcome = heart_failure
Heart failure: 394 patients (23.2%)
Sample of the first 5 patients:


,patient_id,age,sex,prior_heart_failure,diabetes,anterior_mi,heart_failure
0,1,77.0,1,0.0,0.0,1.0,0
1,2,55.0,1,0.0,0.0,4.0,0
2,3,52.0,1,0.0,0.0,4.0,0
3,4,68.0,0,1.0,0.0,0.0,1
4,5,60.0,1,0.0,0.0,4.0,0


### Statistical exploration: building the data profile

The next cell explores every variable and writes one line about each. Together these lines are the *data profile*, and the Data Analyst's lookup tool will hold exactly these lines. For each variable we work out three things:

| For each variable | What it means | Example from our data |
|---|---|---|
| **When it is known** | Some facts are recorded when the patient arrives (age, diabetes). Others only appear later in the stay, such as `pain_relapse_day_2`. To predict at admission, you cannot use the later ones. (UCI: the day 1–3 variables are known only at the end of that day.) | `age`: at admission. `pain_relapse_day_2`: end of day 2. |
| **How much is missing** | The share of patients with no recorded value. A variable that is mostly empty is of little use. | `serum_cpk` is 99.8% missing. |
| **Its link with heart failure, on its own** | How different the heart failure rate (or the average value) is between groups, and whether that gap is bigger than chance. | See below. |

**How the link is measured**
- **Yes/no or coded variables (chi-square test):** compares the heart failure rate across the groups. For `diabetes`, 36.8% of patients with diabetes developed heart failure, against 21.2% of those without. The test gives a p-value (here under 0.001): a gap this big is very unlikely to be chance.
- **Measurements such as age (Welch t-test):** compares the average value in patients with and without heart failure. The average age was 64.85 with heart failure and 60.95 without, and the p-value says whether that gap is more than chance. ("Welch" means the test does not assume both groups are equally spread out.)

**"Unadjusted" means each variable is looked at on its own.** Older patients are more likely to have diabetes, and age itself raises heart failure risk, so part of diabetes's link may really be age. An *adjusted* analysis looks at both together to separate them. The profile cannot do that, so it only shows which variables look worth modelling. Fitting and adjusting a model is what Module 2 does.

In [25]:
import re
from scipy import stats

def pct(x):
    return f"{100 * x:.1f}%"

def p_text(p):
    return "p<0.001" if p < 0.001 else f"p={p:.3f}"

def known_at(name):
    # UCI guidance: the day 1-3 hospital variables are only known at the end of that day
    day = re.search(r"_day_(\d)$", name)
    return f"end of day {day.group(1)}" if day else "admission"

def code_labels(col):
    text = (col.get("codes") or "").split(". ")[0]
    return {int(k): v.strip() for k, v in re.findall(r"(\d+)=([^;]+)", text)}

def association(name):
    # Unadjusted association with heart failure: rates by group (chi-square) or means (Welch t-test)
    col, d = dictionary[name], df[[name, "heart_failure"]].dropna()
    if len(d) < 30:
        return "too little data to compare"
    if col["type"] in ("real", "integer"):
        with_hf, without_hf = d.loc[d["heart_failure"] == 1, name], d.loc[d["heart_failure"] == 0, name]
        p = stats.ttest_ind(with_hf, without_hf, equal_var=False).pvalue
        return f"mean {with_hf.mean():.2f} with HF vs {without_hf.mean():.2f} without ({p_text(p)})"
    groups = d.groupby(name)["heart_failure"].agg(["mean", "size"])
    p = stats.chi2_contingency(pd.crosstab(d[name], d["heart_failure"]), correction=False)[1]
    def rate(code):
        n = int(groups.loc[code, "size"])
        return pct(groups.loc[code, "mean"]) + (f" (n={n})" if n < 30 else "")
    if col["type"] == "binary":
        labels = code_labels(col)
        return f"HF {rate(1)} if {labels.get(1, '1')} vs {rate(0)} if {labels.get(0, '0')} ({p_text(p)})"
    return "HF by code: " + ", ".join(f"{int(c)}: {rate(c)}" for c in groups.index) + f" ({p_text(p)})"

rows = ["| Variable | Type | Known at | Missing | Unadjusted association with heart failure (HF) | Description |",
        "|---|---|---|---|---|---|"]
profile_entries = []      # the same lines as data, for the Data Analyst's lookup tool
for name in features:
    col = dictionary[name]
    desc = col["description"] + (f" ({col['units']})" if col.get("units") else "")
    if col["type"] in ("ordinal", "categorical") and col.get("codes"):
        desc += ". Codes: " + col["codes"].split(". ")[0]
    rows.append(f"| {name} | {col['type']} | {known_at(name)} | {col['stats']['pct_missing']} | {association(name)} | {desc} |")
    profile_entries.append({"name": name, "type": col["type"], "known_at": known_at(name), "missing": col["stats"]["pct_missing"],
                            "meaning": desc, "association": association(name)})
data_profile = "\n".join(rows)
(MATERIALS / "data_profile.md").write_text(data_profile)

# What a good Data Analyst answer should contain
missing = {name: float(dictionary[name]["stats"]["pct_missing"].rstrip("%")) for name in features}
high_missing = [name for name in features if missing[name] > 30]
after_admission = [name for name in features if known_at(name) != "admission"]
review_checklist = f"""# Review checklist: the Data Analyst's answer

A good answer:
- Calls `lookup_variable` for each risk factor and variable it discusses (check the Logs), and quotes numbers that match the data profile exactly. For example, diabetes: {association("diabetes")}.
- Leaves out or flags the variables with more than 30% missing data: {", ".join(high_missing)}.
- Uses only variables known at admission for a prediction made at admission. Known only later: {", ".join(after_admission)}.
- Flags prior_heart_failure (heart failure before this MI): {association("prior_heart_failure")}. A model that uses it may mostly detect heart failure the patient already had.
- Flags cardiogenic_shock_at_admission as a likely competing risk (these patients often die before heart failure can develop): {association("cardiogenic_shock_at_admission")}.
- Says which literature risk factors are not in our dataset, such as ejection fraction, natriuretic peptides (BNP), troponin, kidney function, or modern treatments such as PCI and SGLT2 inhibitors.
- Notes that heart_failure is recorded during the hospital stay only, with no follow-up after discharge.
- Proposes predictive questions with a clear outcome, predictors, and method, and does not claim results that nobody has computed.
"""
(MATERIALS / "review_checklist.md").write_text(review_checklist)

print(f"Data profile: {len(features)} variables. First rows:\n")
print("\n".join(rows[:8]))
print("\n" + review_checklist)

Data profile: 111 variables. First rows:

| Variable | Type | Known at | Missing | Unadjusted association with heart failure (HF) | Description |
|---|---|---|---|---|---|
| age | integer | admission | 0.47% | mean 64.85 with HF vs 60.95 without (p<0.001) | Age of patient (years) |
| sex | binary | admission | 0.00% | HF 20.0% if male vs 28.5% if female (p<0.001) | Sex |
| prior_mi_count | ordinal | admission | 0.24% | HF by code: 0: 22.7%, 1: 23.2%, 2: 27.2%, 3: 22.8% (p=0.691) | Quantity of myocardial infarctions in the anamnesis. Codes: 0=zero; 1=one; 2=two; 3=three or more |
| exertional_angina_history | ordinal | admission | 6.24% | HF by code: 0: 23.3%, 1: 26.7%, 2: 18.2%, 3: 25.6%, 4: 14.5%, 5: 21.6%, 6: 26.8% (p=0.176) | Exertional angina pectoris in the anamnesis. Codes: 0=never; 1=during the last year; 2=one year ago; 3=two years ago; 4=three years ago; 5=4-5 years ago; 6=more than 5 years ago |
| angina_functional_class | ordinal | admission | 4.29% | HF by code: 0: 23.1%, 1

**Write the prompts (run this cell before the n8n steps).** It saves every prompt to `materials/prompts/`; the steps below print them for you to copy.

In [ ]:
N8N_SYSTEM = '''You are the Literature Scout on a research team studying heart failure after acute myocardial infarction (MI).

Tools:
- search_pubmed: searches PubMed and returns PubMed IDs (PMIDs).
- fetch_abstracts: given comma-separated PMIDs, returns titles and abstracts.

How to work:
1. Turn the request into a short PubMed query, for example: "Myocardial Infarction"[MeSH] AND "Heart Failure"[MeSH].
2. Search, then fetch the abstracts of the 5 most relevant PMIDs in one call (fewer only if fewer than 5 are relevant). If a search returns nothing, broaden the query and try again.
3. Summarize each paper in 2-3 sentences under its own heading: design, population, main finding. Write your answer in Markdown.
4. Cite every claim with its PMID. Never cite a PMID you did not retrieve with a tool.
5. Add one line headed "What this means for our team".
6. End with a "References" list of every paper you fetched, one per line: title, journal, year, PMID, and the link https://pubmed.ncbi.nlm.nih.gov/<PMID>/

Formatting: give each paper and each risk factor its own ### heading, and use bullet points under it. Do not use numbered lists anywhere in your answer.'''

N8N_REPORT_TEMPLATE = '''<style>
body { font-family: -apple-system, "Segoe UI", Arial, sans-serif; max-width: 960px; margin: 40px auto; padding: 0 16px; line-height: 1.55; color: #1f2933; }
h1 { border-bottom: 3px solid #1f6feb; padding-bottom: 6px; }
h2 { margin-top: 2em; border-bottom: 1px solid #d0d7de; padding-bottom: 4px; color: #0b3d91; }
h3, h4 { color: #0b3d91; }
code { font-family: Menlo, Consolas, monospace; font-weight: bold; font-size: 0.88em; background: #eef2f7; padding: 1px 5px; border-radius: 4px; }
table { border-collapse: collapse; width: 100%; margin: 1em 0; font-size: 0.9em; }
th { background: #0b3d91; color: #ffffff; text-align: left; }
th, td { border: 1px solid #d0d7de; padding: 6px 10px; vertical-align: top; }
tr:nth-child(even) td { background: #f6f8fa; }
a { color: #1f6feb; }
hr { border: none; border-top: 1px solid #d0d7de; margin: 2em 0; }
@media print { body { margin: 0; max-width: none; } tr { page-break-inside: avoid; } th, tr:nth-child(even) td { -webkit-print-color-adjust: exact; print-color-adjust: exact; } }
</style>

# Heart failure after MI: literature and data report

- **Question:** {{ $('When chat message received').first().json.chatInput }}
- **Date:** {{ $now.toFormat('yyyy-MM-dd') }}

## Part 1. What the literature says (Literature Scout)

{{ $('Literature Scout').first().json.output }}

## Part 2. What our data can support (Data Analyst)

{{ $json.output }}'''

N8N_SCHEDULED_PROMPT = '''Prepare this week's Literature Watch digest.
Find papers published in the last 7 days on heart failure after acute myocardial infarction.
Choose the 5 most relevant and write the digest in the format from your instructions.'''

N8N_ANALYST_SYSTEM = f'''You are the Data Analyst on a research team studying heart failure after acute myocardial infarction (MI).
You receive the team's research question and the Literature Scout's findings. Your job is to check those findings against our own patients and to plan predictive analyses that our dataset can support.

Our dataset: {len(df):,} patients admitted with acute MI to one hospital in Krasnoyarsk, Russia, 1992-1995.
Outcome: heart_failure, chronic heart failure during the hospital stay, in {n_hf} patients ({pct(n_hf / len(df))}). There is no follow-up after discharge.

Tool:
- lookup_variable: give it a variable name or a keyword. It returns the matching variables with their type, when each is known (at admission, or at the end of hospital day 1, 2, or 3), how much is missing, what it means, and its unadjusted association with heart failure. "No variable matches" means our dataset does not have it. It only looks up results computed in advance; it cannot calculate anything new.

How to work:
1. For each risk factor from the Literature Scout, call lookup_variable with its name or a keyword. If several variables could stand for it, look at the candidates. If nothing matches, it is "Not in our dataset".
2. Look up any other variable you want to recommend.
3. Quote numbers only from the tool's results, exactly. Never estimate or invent numbers.

Write your answer in Markdown with these sections:
1. Literature vs. our data: a table with one row per risk factor from the Literature Scout: Risk factor | Our variable(s) | Missing | Result from lookup_variable | Usable? (Yes / Proxy / Not in our dataset).
2. Candidate feature sets: two or three sets of variables for predicting heart_failure, for example one using only information known at admission and one that adds the first three days. For each, list the variables and explain the choice in one or two sentences.
3. Data issues: variables to leave out or treat with care, and why: more than 30% missing, very small groups, known only after the prediction time, or so close to the outcome that they could make a model look better than it is.
4. Predictive questions we can answer: three to five specific questions, each with its outcome, predictors, and a suitable method (for example logistic regression).
5. Questions we cannot answer: two or three, with the reason (for example, a variable that is not in our dataset).

Rules:
- Write every variable name in backticks, for example `diabetes` or `prior_heart_failure`, everywhere, including inside tables.
- Inside the sections use bullet points, not numbered lists: numbered lists restart at 1 in the report.
- Unadjusted associations are a starting point, not proof that a variable predicts heart failure. Do not choose features by p-value alone.
- Patients who died early had little time to develop heart failure, so very severe conditions (such as cardiogenic shock) can look protective.
- Keep it concise.'''

N8N_LOOKUP_TOOL_DESCRIPTION = '''Look up variables in our MI patient dataset (1,700 patients). Input a variable name or a keyword, for example diabetes, potassium, or prior_heart_failure. Returns up to 8 matching variables with: type, when it is known (admission or later), % missing, meaning, and its precomputed unadjusted link with heart failure. Returns "No variable matches" if the dataset does not have it. It only looks up results computed in advance; it cannot calculate anything new.'''

PROFILE_TOOL_JS = r'''// lookup_variable: looks up our precomputed data profile (Section 1). It does not calculate anything.
const PROFILE = [
''' + ",\n".join(json.dumps(entry, ensure_ascii=False) for entry in profile_entries) + r'''
];

const input = String(query || '').trim();
if (!input) return 'Give a variable name or a keyword, for example "diabetes" or "potassium".';

const wanted = input.toLowerCase().replace(/[_-]+/g, ' ');
const words = wanted.split(/\s+/).filter(Boolean);
const exact = PROFILE.filter(p => p.name.replace(/_/g, ' ') === wanted);
const hits = exact.length ? exact : PROFILE.filter(p => {
  const text = (p.name.replace(/_/g, ' ') + ' ' + p.meaning).toLowerCase();
  return words.every(w => text.includes(w));
});

if (!hits.length) return `No variable matches "${input}" in our dataset. Try a shorter keyword. If it still finds nothing, the dataset does not have it.`;

const shown = hits.slice(0, 8);
const header = hits.length > shown.length ? `${hits.length} variables match; showing the first ${shown.length}:` : `${hits.length} variable(s) match:`;
return header + '\n' + shown.map(p => `${p.name} | ${p.type} | known at ${p.known_at} | ${p.missing} missing | ${p.meaning} | ${p.association}`).join('\n');'''

N8N_SCOUT_MCP_SYSTEM = '''You are the Literature Scout on a research team studying heart failure after acute myocardial infarction (MI).
Today's date is {{ $now.toFormat('yyyy/MM/dd') }}.

Tools (from a PubMed server):
- pubmed_search_articles: searches PubMed and returns PubMed IDs (PMIDs). Useful inputs: query (PubMed syntax, MeSH terms allowed), maxResults, sort, and dateRange.
- pubmed_fetch_articles: given a list of PMIDs, returns titles, abstracts, journals, and more.

How to work:
1. Turn the request into a short PubMed query, for example: "Myocardial Infarction"[MeSH] AND "Heart Failure"[MeSH]. Use dateRange only when the request names a time window. It needs BOTH minDate and maxDate (YYYY/MM/DD), so work them out from today's date.
2. Search, then fetch the 5 most relevant PMIDs in one call (fewer only if fewer than 5 are relevant). If a search returns nothing, broaden the query and try again.
3. Summarize each paper in 2-3 sentences under its own heading: design, population, main finding. Write your answer in Markdown.
4. Cite every claim with its PMID. Never cite a PMID you did not retrieve with a tool.
5. Add one line headed "What this means for our team".
6. End with a "References" list of every paper you fetched, one per line: title, journal, year, PMID, and the link https://pubmed.ncbi.nlm.nih.gov/<PMID>/

Formatting: give each paper and each risk factor its own ### heading, and use bullet points under it. Do not use numbered lists anywhere in your answer.'''

N8N_ANALYST_USER = '''Research question: {{ $('When chat message received').first().json.chatInput }}

Literature Scout findings (approved by the PI):
{{ $('Literature Scout').first().json.output }}'''

N8N_ANALYST_USER_SCHEDULED = '''Research question: Which patients develop heart failure after acute MI, and how could we predict it?

Literature Scout findings (approved by the PI):
{{ $('Literature Scout').first().json.output }}'''

N8N_HUMAN_REVIEW_MESSAGE = '''The Literature Scout has finished. Please review its findings before the Data Analyst starts.

**Check that:**
- the papers are relevant to heart failure after MI,
- every PMID comes with a title that you can find on PubMed,
- the risk factors the Scout lists are the ones we want to test in our own data.

Choose **Approve** to continue to the Data Analyst, or **Decline** to stop here.

---

{{ $('Literature Scout').first().json.output }}'''

N8N_NOT_APPROVED_MESSAGE = '''🛑 **Workflow stopped.** The Literature Scout's findings were not approved, so no report was created. Change your question or the Scout's instructions, and ask again.'''

N8N_REPORT_READY_MESSAGE = '''✅ **Workflow completed.** The team has finished. The Data Analyst's answer is below.

**To save the report as a PDF:** open the **Convert to File** node in the Logs panel, click **Download**, open the file in your browser, and choose File → Print → Save as PDF.

---

{{ $('Data Analyst').first().json.output }}'''

prompt_files = {
    "n8n_scout_system.txt":            N8N_SYSTEM,
    "n8n_human_review_message.txt":    N8N_HUMAN_REVIEW_MESSAGE,
    "n8n_not_approved_message.txt":    N8N_NOT_APPROVED_MESSAGE,
    "n8n_analyst_system.txt":          N8N_ANALYST_SYSTEM,
    "n8n_lookup_tool_description.txt": N8N_LOOKUP_TOOL_DESCRIPTION,
    "n8n_profile_tool.js":             PROFILE_TOOL_JS,
    "n8n_scout_mcp_system.txt":        N8N_SCOUT_MCP_SYSTEM,
    "n8n_analyst_user.txt":            N8N_ANALYST_USER,
    "n8n_report_template.md":          N8N_REPORT_TEMPLATE,
    "n8n_report_ready_message.txt":    N8N_REPORT_READY_MESSAGE,
    "n8n_scheduled_prompt.txt":        N8N_SCHEDULED_PROMPT,
    "n8n_analyst_user_scheduled.txt":  N8N_ANALYST_USER_SCHEDULED,
}
for name, text in prompt_files.items():
    (PROMPTS / name).write_text(text)
    print(f"wrote {PROMPTS / name}  ({len(text):,} characters)")

---
## 2. Build the team in n8n

[n8n](https://n8n.io) is a workflow automation platform: you connect nodes on a canvas to call APIs, process data, and run on schedules. Its **AI Agent** node puts an LLM inside such a workflow as a decision-maker. The *workflow* decides when each step runs; each *agent* decides which tools to call.

**Why n8n for this?** A literature watch should run *by itself*, on a schedule, with fixed steps around the AI (here a human approval and a report). n8n is built for triggers, integrations, and fixed steps around agents.

### Shortcut: import a ready-made workflow

You do not have to build the team by hand. Two finished workflows are in the `workflows/` folder, so you can have the whole team running in a few minutes:

| File | What you get |
|---|---|
| `workflows/mi_research_team.json` | The full team: Literature Scout (PubMed tools built from API calls) → Human Review → Data Analyst (with its `lookup_variable` tool) → report |
| `workflows/mi_research_team_mcp.json` | The same team, but the Scout gets its PubMed tools from an existing MCP server (see *Variant* below) |

**To import a workflow**
1. In n8n, create a new workflow.
2. Open the **⋯** menu at the top right, choose **Import from File**, and select the `.json` file.
3. Attach a chat model to **each of the two agents**, the Literature Scout and the Data Analyst: click the *Chat Model* connector on the agent, pick your provider, and select your credential. The model must support tool calling. Imported files contain no keys or credentials, so a red warning on an agent after import usually just means the chat model is missing.
4. Open the chat panel and ask your question. When the Human Review message appears, click **Approve**. When the Data Analyst finishes, open **Logs → Convert to File → Download** to get `report.html`.

Importing is the fast way to have it running, and it gives you something to compare your own build with. Building it step by step (below) is still the best way to learn where each part lives.


### Core concepts

| Concept | What it is | In our workflow |
|---|---|---|
| **Trigger** | Starts the workflow | Chat Trigger (Schedule Trigger for the weekly run) |
| **AI Agent** | An LLM that reasons and calls tools in a loop | Literature Scout, Data Analyst |
| **Tool** | A capability the agent may choose to use | Scout: two HTTP Request tools that call PubMed. Data Analyst: a Code Tool that looks up our data profile |
| **Expression** (`{{ ... }}`) | Inserts data from earlier nodes into a field | Passing the Scout's answer on; `$fromAI()` lets the model fill in a parameter |
| **Human in the loop** | A step that pauses until a person responds | The Chat node's *Send and Wait for Response*, then an **If** node to branch |
| **Regular nodes** | Fixed steps with no model involved | Markdown and Convert to File build the report |

---

### Look at the tool before the agent uses it

A tool is just a web address (an *API call*) plus a description. Open this link in your browser:

[https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&retmode=json&term=heart+failure+myocardial+infarction](https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&retmode=json&term=heart+failure+myocardial+infarction)

You will see a list of PubMed IDs (`"idlist"`): exactly what the agent's `search_pubmed` tool receives. The link has a URL (which service) and query parameters (the `name=value` pairs after the `?`). In n8n, the URL goes in the **URL** field and each `name=value` pair becomes one row under **Query Parameters**.

The parameter names come from NCBI's [E-utilities reference](https://www.ncbi.nlm.nih.gov/books/NBK25499/), not from the PubMed website. These are the ones we use:

| Parameter | Meaning |
|---|---|
| `db=pubmed` | Which NCBI database to search |
| `term` | The search query, as you would type it in PubMed |
| `retmode` | Reply format: `json` or `text` |
| `retmax=10`, `sort=relevance` | At most 10 results, best matches first |
| `datetype=pdat`, `reldate` | Only papers published in the last `reldate` days |
| `id`, `rettype=abstract` | (`fetch_abstracts`) the PMIDs to fetch, and return their abstracts |

The cell below makes the same two calls from Python.

In [27]:
import os
import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
NCBI = {"tool": "insilico_team_tutorial", "email": "you@example.org"}   # NCBI asks callers to identify themselves
if os.getenv("NCBI_API_KEY"):
    NCBI["api_key"] = os.environ["NCBI_API_KEY"]

def search_pubmed(term, days=None, retmax=5):
    params = {"db": "pubmed", "term": term, "retmode": "json", "retmax": retmax, "sort": "relevance", **NCBI}
    if days:
        params.update({"reldate": days, "datetype": "pdat"})
    r = requests.get(f"{EUTILS}/esearch.fcgi", params=params, timeout=20)
    r.raise_for_status()
    return r.json()["esearchresult"]["idlist"]

def fetch_abstracts(pmids):
    params = {"db": "pubmed", "id": ",".join(pmids), "rettype": "abstract", "retmode": "text", **NCBI}
    r = requests.get(f"{EUTILS}/efetch.fcgi", params=params, timeout=20)
    r.raise_for_status()
    return r.text

ids = search_pubmed('"Myocardial Infarction"[MeSH] AND "Heart Failure"[MeSH]', days=90)
print("Tool 1 (search_pubmed) returns PMIDs:", ids)
print("\nTool 2 (fetch_abstracts) returns text like this:\n")
print(fetch_abstracts(ids[:2])[:1500] if ids else "No results. Try a broader query or a longer time window.")

Tool 1 (search_pubmed) returns PMIDs: ['42645558', '42185090', '42484668', '42558062', '42612493']

Tool 2 (fetch_abstracts) returns text like this:

1. Intensive Care Med. 2026 Oct;52(10):2160-2177. doi:
10.1007/s00134-026-08574-4.  Epub 2026 Aug 26.

State-of-the-art review: ß-blockade in critical illness.

Ostermann M(1), De Backer D(2), Belley-Cote E(3), Cecconi M(4)(5), Chew MS(6), 
Dionne JC(7), Hollenberg SM(8), Jakob SM(9), Kanji S(10), Leach R(11), McPhail 
MJ(12)(13), Monnet X(14), Morelli A(15), Singer M(16), Wilkman E(17), Citerio 
G(18)(19).

Author information:
(1)Department of Intensive Care, King's College London, Guy's & St Thomas' NHS 
Foundation Trust, London, UK. m.ostermann@nhs.net.
(2)Department of Intensive Care, CHIREC Hospitals, Université Libre de 
Bruxelles, Brussels, Belgium.
(3)Divisions of Cardiology and Critical Care, McMaster University, Hamilton, 
Canada.
(4)Department of Anaesthesia, Intensive Care, IRCCS-Humanitas Research Hospital, 
Rozzano, Italy.
(

---

### Build the team

```
Chat Trigger ─► Literature Scout ─► Human Review ─► Human Decision ─ approved ─► Data Analyst ─► Markdown ─► Convert to File ─► Report ready
                 ├─ Chat Model       (Chat node)     (If)            │            ├─ Chat Model   (to HTML)   (report.html)     (Chat node)
                 ├─ Tool: search_pubmed  Approve / Decline           │            └─ Tool: lookup_variable (Code Tool)
                 └─ Tool: fetch_abstracts                            └ declined ─► Not approved (Chat node)
```

> **Short on time?** Import the finished workflow instead (see *Shortcut: import a ready-made workflow* above), attach a chat model to each agent, and use the steps below to understand each part.

#### Stage A — The Literature Scout

**Step 1.** Create a new workflow and name it *MI Research Team*.

**Step 2.** Add a **Chat Trigger**. It gives you a chat panel to talk to the team.

**Step 3.** Add an **AI Agent** and connect it to the trigger. Rename it **Literature Scout** (the report finds its answer by this name). Under *Options → System Message*, replace the default text with the Scout prompt printed below.

**Step 4.** Click **+** under *Chat Model*, pick your provider, and add your API key as a credential. 

**Step 5 — Tool 1, `search_pubmed`.** Click **+** under *Tool* and add an **HTTP Request Tool**:

| Field | Value |
|---|---|
| Name | `search_pubmed` |
| Description | Search PubMed for biomedical articles. Input a PubMed query string and the number of days to look back. Returns a JSON list of PubMed IDs (PMIDs). |
| Method / URL | `GET` · `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi` |
| Send Query Parameters | On |

Add seven query parameters. The first five are fixed; the last two are filled in by the model on every call, and **the tool returns nothing without `term`**.

| Name | Value |
|---|---|
| `db` | `pubmed` |
| `retmode` | `json` |
| `retmax` | `10` |
| `sort` | `relevance` |
| `datetype` | `pdat` |
| `term` | `{{ $fromAI('term', 'PubMed query string; MeSH terms allowed', 'string') }}` |
| `reldate` | `{{ $fromAI('days', 'Only include papers published in the last N days', 'number', 365) }}` |


> For `term` and `reldate`, click *"Let the model define this parameter"*, or type the expression exactly (two curly braces on each side; the field must show as an **Expression**). This is where the **model** decides the arguments.

**Step 6 — Tool 2, `fetch_abstracts`.** Add a second **HTTP Request Tool**. Without it the agent gets only PMIDs, not abstracts.

| Field | Value |
|---|---|
| Name | `fetch_abstracts` |
| Description | Fetch titles and abstracts for comma-separated PubMed IDs returned by search_pubmed. |
| Method / URL | `GET` · `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi` |
| Query parameters | `db` = `pubmed`, `rettype` = `abstract`, `retmode` = `text`, `id` = `{{ $fromAI('pmids', 'Comma-separated PubMed IDs', 'string') }}` |

**Step 7 — Test the Scout.** In the chat, try:
- *"What has been published in the last year on heart failure after myocardial infarction?"*
- *"Is there trial evidence that SGLT2 inhibitors prevent heart failure after MI? If there are few trials, look at observational studies instead."* Watch whether it runs more than one search and changes its query.

Open the **Logs** panel to see every tool call, with the arguments the model chose and the result it received (the *reason → act → observe* loop). Click `search_pubmed` or `fetch_abstracts` to read the raw PMIDs and abstracts. If nothing is found, `"Empty term and query_key"` in the `search_pubmed` output means `term` is missing. The Scout fetches only the 5 most relevant of up to 10 results; ask in the chat for the rest.

#### Stage B — The Human Review

**Step 8.** In the Chat Trigger, add *Options → Response Mode* = **Using Response Nodes**. From now on the chat shows only what **Chat nodes** send (this is what lets a Chat node pause and ask you a question), so the Scout's answer will appear inside the approval message.

**Step 9.** Click **+** after the Literature Scout and add a **Chat** node. Rename it **Human Review** and set:
- *Operation:* **Send and Wait for Response**
- *Message:* the Human Review message printed below
- *Response Type:* **Approval**, with *Approval Options*: **Type of Approval** = **Approve and Disapprove**, **Approve Button Label** = `Approve`, **Disapprove Button Label** = `Decline`

**Step 10.** After the Human Review, add an **If** node, rename it **Human Decision**, and set *Value 1* = `{{ $json.data.approved }}`, type **Boolean**, operator **is true**. (`approved` is `true` after Approve and `false` after Decline.)
- The **true** output (approved) leads to the Data Analyst (Step 12).
- On the **false** output (declined), add a **Chat** node named **Not approved**, with *Operation* **Send Message** and the "not approved" text printed below.

**Step 11 — Test both paths.** Send a question. When the Scout finishes, the chat shows its findings with **Approve** and **Decline** buttons, and the run *waits* (it shows as *Waiting* in Executions). Decline: the chat says "Workflow stopped". Approve: nothing more happens yet.

As the PI, read the findings as you would a colleague's: are the papers relevant, does every PMID have a matching title on PubMed, and are these the risk factors we want to test in our data? If not, decline.

#### Stage C — The Data Analyst

**Step 12.** Click **+** on the **true** output of *Human Decision*, add an **AI Agent**, and rename it **Data Analyst**:
- *Source for Prompt (User Message):* **Define below**, with the Data Analyst user prompt printed below. It reads the Scout's answer by node name (`{{ $('Literature Scout').first().json.output }}`), because the node before it now outputs only the approval result.
- *Options → System Message:* the Data Analyst system prompt printed below. It describes the dataset and the tool, and contains no numbers.
- *Chat Model:* attach one (copy the Scout's).
- *Tool:* click **+** under *Tool* and add a **Code Tool**. This tool is not an API call: it runs a little code that we give it.

| Field | Value |
|---|---|
| Name | `lookup_variable` |
| Description | The description printed below |
| Language | JavaScript |
| JavaScript | The code printed below (about 30 KB: our data profile from Section 1, one entry per variable, and a small search). Copy everything in `materials/prompts/n8n_profile_tool.js` |

The tool **looks up** numbers we computed in advance; it calculates nothing. The agent decides which variables to look up (see *Limits of this workflow* below).

**Step 13 — Test the team.** Ask: *"What are the main risk factors for heart failure after acute MI, and which of them could we use to predict it in our patients?"* Approve the findings, then open **Logs → Data Analyst**. You will see its `lookup_variable` calls: the keyword the model chose and what came back (the chat will show its answer after Stage D). Compare with the review checklist: does every number match a lookup result and `materials/data_profile.md`? Did it flag variables with a lot of missing data, variables known only after admission, and risk factors our dataset lacks (a *No variable matches* result)?

#### Stage D — The report, and running it every week

**Step 14 — Report.** n8n cannot create PDFs itself, so we build a web page (HTML) and save it as a PDF from the browser.
1. **Markdown** node after the Data Analyst: *Mode* = Markdown to HTML; *Markdown* = the report template printed below (its `<style>` block sets the look, including bold variable names; edit it to change colors); *Options*: switch on **Complete HTML Document**, **Automatic Linking to URLs**, and **Tables Support**.
2. **Convert to File** node: *Operation* = Convert to Text File; *Text Input Field* = `data`; *Options → File Name* = `report.html`.
3. **Chat** node named **Report ready**: *Operation* = **Send Message** and the "report ready" text printed below. It shows the analysis in the chat.
4. Run the whole workflow and approve. Then open **Logs → Convert to File → Download**, open the file in your browser, and choose **File → Print → Save as PDF**.

**Step 15 (optional) — Run it every week.** Duplicate the workflow, then:
- Replace the Chat Trigger with a **Schedule Trigger**, and use the scheduled prompts printed below for the Scout and the Data Analyst.
- Chat nodes need a Chat Trigger, so replace **Human Review** with a **Send Email** node set to **Send and Wait for Response** (response type **Approval**): the PI approves from an email link. Keep *Human Decision* as it is, delete **Not approved** and **Report ready**, and delete the `Question:` line in the report template.
- Switch the workflow on (the toggle at the top of the editor).

In [ ]:
print("=== Step 3: Literature Scout → Options → System Message ===\n")
show("n8n_scout_system.txt")
print("\n\n=== Step 9: Human Review → Message ===\n")
show("n8n_human_review_message.txt")
print("\n\n=== Step 10: Not approved → Message ===\n")
show("n8n_not_approved_message.txt")
print("\n\n=== Step 12: Data Analyst → Source for Prompt: Define below ===\n")
show("n8n_analyst_user.txt")
print("\n\n=== Step 12: Data Analyst → Options → System Message ===\n")
show("n8n_analyst_system.txt")
print("\n\n=== Step 12: Data Analyst → Tool lookup_variable → Description ===\n")
show("n8n_lookup_tool_description.txt")
print("\n\n=== Step 12: Data Analyst → Tool lookup_variable → JavaScript (first lines; copy ALL of materials/prompts/n8n_profile_tool.js) ===\n")
tool_code = (PROMPTS / "n8n_profile_tool.js").read_text().splitlines()
print("\n".join(line[:150] for line in tool_code[:5]) + f"\n... ({len(tool_code)} lines in all)")
print("\n\n=== Step 14: Markdown node → Markdown ===\n")
show("n8n_report_template.md")
print("\n\n=== Step 14: Report ready → Message ===\n")
show("n8n_report_ready_message.txt")
print("\n\n=== Step 15: scheduled version → Literature Scout prompt (Define below) ===\n")
show("n8n_scheduled_prompt.txt")
print("\n\n=== Step 15: scheduled version → Data Analyst prompt (Define below) ===\n")
show("n8n_analyst_user_scheduled.txt")

---

### Variant: the same team with an existing MCP server

So far the Scout's PubMed tools are ones we built ourselves from API calls. There is another way: connect to a server that already offers ready-made tools. This uses **MCP (Model Context Protocol)**, a standard way for an agent to discover and use tools that somebody else runs. Here we use a public PubMed MCP server.

To use the workflow with MCP Tools:  In n8n, create a workflow, open **⋯ → Import from File**, and select `workflows/mi_research_team_mcp.json`. Attach a chat model to each agent, then compare it with the workflow you built. Download the file and play with it.

**To change your own workflow instead:** delete the Scout's two HTTP Request tools and add one **MCP Client Tool** to the Literature Scout:

| Field | Value |
|---|---|
| Endpoint | `https://pubmed.caseyjhand.com/mcp` |
| Server Transport | HTTP Streamable |
| Authentication | None |
| Tools to Include | Selected: `pubmed_search_articles` and `pubmed_fetch_articles` |

n8n asks the server which tools it has and what inputs they take, so there is nothing to type: no URL, no query parameters, no `$fromAI()`. The server offers 12 tools; we select 2 so the agent stays focused. Then replace the Scout's system message with the version below. The tools have different names, and the server's date filter needs both a start and an end date, so the prompt gives the agent today's date.

In [ ]:
print("=== Literature Scout (MCP version) → Options → System Message ===\n")
show("n8n_scout_mcp_system.txt")

**What to compare in the Logs.** Run the same question through both Scouts. Which tools were called, and with what arguments? How many calls did each make? How long is each tool result? n8n puts the node's name in front of each MCP tool name, for example `PubMed_MCP_pubmed_search_articles`. (One fetched article is about 11,000 characters, so the Scout fetches a few at a time.) In one of our test runs the MCP Scout made 2 tool calls and the HTTP Scout 7; your numbers will differ from run to run.

**Good to know**
- **It is somebody else's server.** This one is a public server run by an individual. It could change, slow down, or go away, and its page states no terms or rate limits. Your queries go to that server. That is why the from-scratch workflow stays the main path, and the MCP version is the quick alternative.
- **We use 2 of its 12 tools on purpose.** Fewer tools keep the agent focused. As an exercise, add `pubmed_lookup_mesh` and see whether the Scout builds better queries.
- **MCP tools are still tools.** The agent sees a name, a description, and inputs, exactly as for the tools you built.

---

### Discussion: do you need an agent for this?

A few lines of code can call the PubMed API. Why build an agent? For a fixed search, that objection is right:

> **If you can write down every step in advance, write code. If the next step depends on what you find, use an agent.**

| Task | A script | The agent |
|---|---|---|
| Run the same search every week | ✅ Cheaper and repeatable | Overkill |
| Turn a free-text question into a PubMed query | Needs a person | ✅ |
| Notice 0 results and broaden the query | Only if coded in advance | ✅ |
| Pick the relevant papers and summarize them with citations | Cannot judge relevance | ✅ |

Not everything here is an agent. The Data Analyst is an agent with a deliberately simple tool: it decides what to look up, but the tool only returns numbers we computed in advance. The weekly version (Step 15) is a *workflow with LLM steps*; Markdown and Convert to File involve no model; and the Human Review is a fixed checkpoint. Where a human must stay in the loop is a design choice: here it is before anything builds on the Scout's papers.

---

### Limits of this workflow: read this before trusting the output

- **The Data Analyst looks things up; it calculates nothing.** Its tool, `lookup_variable`, returns numbers that were computed in Python in Section 1 (missing %, associations, p-values) and stored in the tool. The agent chooses what to look up and quotes the results. It cannot run a new analysis, or look at a variable or combination we did not include.
- **Everything it knows is unadjusted and one variable at a time.** It cannot adjust for age or sex, or fit a model. That is where Module 2 starts.
- **It can still get things wrong.** A language model may pick the wrong variable for a risk factor, misquote a result, or skip a caveat. In the Logs, every number in its answer should match a `lookup_variable` result; compare with the review checklist and the data profile.
- **The Literature Scout reads titles and abstracts, not full papers,** and is only as good as its search. The Human Review is the safeguard.
- **The tool's answers are only as good as what we precomputed.** Moving the profile out of the prompt keeps the prompt short (under 3,000 characters instead of 22,000) and makes every number traceable to a tool call, but it does not make the numbers any smarter.

**Extensions to try later**
- **Give the Data Analyst a calculation tool.** `lookup_variable` returns results computed in advance. A tool that *computes* on demand could answer new questions, such as a different cutoff or a new subgroup. Three ways, from simpler to more realistic (we have not built them here):
  - **A. A data table and a small tool workflow (n8n only).** Upload the CSV into an n8n *data table*. A second workflow reads it and runs the statistics, and the agent calls that workflow with the **Call n8n Workflow Tool**. Works with Docker and n8n Cloud.
  - **B. A Python MCP server.** A small local server exposes tools (like the Module 2 tools `compare_groups` and `train_model`), and n8n connects with the **MCP Client Tool**. A real Python stack, but it needs a running server, and n8n Cloud cannot reach your computer.
  - **C. One Code Tool node.** A single JavaScript tool with the data embedded. The least setup, but the data is hidden inside code, and the code sandbox may be limiting.

  Even with a tool, the agent can only do what the tool was coded to do, for example unadjusted comparisons. Fitting and adjusting models is Module 2.
- **Let the human steer, not just stop.** With *Approval*, *Decline* can only end the run. Use the **Free Text** response type instead, so the human can write a comment (for example "focus on randomized trials"), pass it back to the Literature Scout as extra instructions, and show the new findings for approval again. This needs a loop in the workflow; we have not built it here.
- **Approve each tool call.** n8n can pause for a person *before* an agent runs a particular tool, for example every PubMed search (see n8n's documentation on human in the loop for AI tool calls).
- **Explore more MCP servers.** The PubMed MCP variant above is one example. MCP servers exist for other biomedical sources, and Module 3 returns to MCP.

---
## 3. Where the canvas runs out, and what comes next

| What we needed | On the canvas | Session 2b |
|---|---|---|
| A real analysis | The Data Analyst looked up results computed in advance; it could not calculate anything new | The agent runs code on the full dataset and fits the models it proposed |
| Adjusted results | Only unadjusted, one-variable associations | Multivariable models with proper validation |
| Steering the agents | The human can only approve or stop | Comments are passed back to the agent in code |
| Trustworthy numbers and citations | We checked by eye against the logs and the checklist | Checked automatically in code |
| Evaluation and reproducibility | A few questions by hand; workflows exported as large JSON files | Many runs, scored; agents in plain Python under version control |

**Session2b rebuilds the Data Analyst in Python with an agent SDK** and answers one of the predictive questions proposed here.

### Discussion questions
1. Which parts of your own research workflow would you hand to a Literature Scout, and which would you never automate?
2. What would make you decline the Scout's findings as the PI?
3. The Data Analyst was told to quote only numbers from its tool's results. Did it? What does that say about relying on prompts?